### **Convolutional Neural Network (CNN)**
**CNN** is a specialized deep-learning **ANN** architecture that is designed for image processing, computer vision, and spatial data analysis. It is used for object detection, face recognition, self-driving, etc...

**CNN** = **Convolutional Network** + **Pooling** + **ANN**


**CNN** is nothing but ANN with feature engineering. The first thing we do is **we extract features** from the image, **compress the image features** and then send the feaures to the ANN for processing.

**Why ANN alone is not good enough for image processing?**
- Say you have a color image of 50 x 50 pixels

- A color is combination 3 channels: R G B.

- Hence, the image is made of 50 x 50 x 3 = 7500 pixels.

- But pixels alone does not identify the object in the image. **You need spatial features ALSO.**
  - An image is basically a grid of pixels.
  - Spatial features capture where things are and how they are captured in the grid.
  - Examples:
    - Edges (where the intensity of pixels changes sharply)
    - Corners (where two Edges meet)
    - Textures (Patterns in Pixel intensity)
    - Shapes / Contours (arrangement of pixels forming objects)
  - CNNs are great at learning spatial features using CNN filters because filters slide across the images and detect these local patterns.

- A 3 minutes video captured using 60Hz camera gives you nearly 10000 frames or images.

- You end up having **10000 x 7500 = 75, 000, 000 pixels** which is  huge data to process a 3 minute video.

- **_Hence, along with Spatial features, you also need data compression or sample downsizing._***

- This explains why ANN alone is not enough for image processing.

#### **Solution**
You need **Convolution Network** and **Pooling**. **Convolution Network** extracts the spatial features from the image, and **Pooling** does the sample or feature downsizing.

![CNN](https://media.springernature.com/full/springer-static/image/art%3A10.1038%2Fs41598-024-51258-6/MediaObjects/41598_2024_51258_Fig1_HTML.png?as=webp)

https://poloclub.github.io/cnn-explainer/


### 1. Convolution Network / Layer
**A convolution layer is the core building block of a Convolutional Neural Network (CNN)**. It is designed to automatically extract spatial features from input data (usually from images) using a set of learnable filters or kernels.

**Imagine**

You have an image represented as a grid of pixel values (e.g., 7x7 color scale. means 3 channels ( R G B) of 7x7 pixels). The convolution layer slides a small filter (like 3x3 ) over the image and performs element-wise multiplication and summation to produce a feature map (also called an activation map).

![CNN](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*ciDgQEjViWLnCbmX-EeSrA.gif)

Popular Kernel Sizes are:
- 7 x 7 (used usually in the first convolution layer for large images)
- 5 x 5
- 3 x 3

**even number kernel is not used because it fails to capture the center of the image.

**Note**
- ***One kernel / filter mean one feature***. Hence, more feature you need, more kernel needs to be added.
- If we need minute details from the image, we will have to apply multiple filters. (Popular filter sizes are 16, 32, and 64.
- More feature, more complex is the model.
- **_An activation function applied after convolution to introduce non-linearity, which helps the network learn complex patterns._**

**Other Parameters in Convolutional Operation**

- **Stride**: Stride indicates how many pixels the kernel should be shifted over at a time.The impact stride has on a CNN is similar to kernel size.
When stride is set to 1, the filter moves across one pixel at a time, and when the stride is set to 2, the filter moves across two pixels at a time.
The higher the stride value, the smaller the output, and vice versa.

- **Padding**: Padding is often necessary when the kernel extends beyond the activation map.



### 2. Pooling
**Pooling** is a downsampling operation used in Convolutional Neural Networks (CNNs) to reduce the spatial size (width × height) of feature maps, while retaining important features. It is done to decrease computational load and control overfitting.

#### Max Pooling vs Average Pooling

![Pooling](https://cdn.analyticsvidhya.com/wp-content/uploads/2024/08/597371-kqieqhxzicu7thjaqbfpbq-66c7045e59b1e.webp)



### 3. Fully Connected (Dense) Layers
Dense layer is nothing, but the ANN layer. **After several** convolutional and pooling layers, the feature maps are flattened and passed to fully connected layers for classification or regression.

### 4. Dropout (optional)
Dropout is a regularization technique used during training to prevent overfitting by randomly "dropping out" (turning off) a fraction of the neurons in a layer during each training step.

## A Simple CNN Code - Training on MNIST Digits - Using Pipeline

In [2]:
!pip install -U scikeras scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 101.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
# Step 1: Imports
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from matplotlib import pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from scikeras.wrappers import KerasClassifier
from sklearn.metrics import accuracy_score

# -----------------------------------------------------------
# Load and preprocess the data
# x_train is a 3 dimensional array 60000 x 28 x 28 --> 60000 training images. Each image is of "28 x 28" pixel image showing hand-written digit sample.
# Each element in the x_train (i.e. x_train[][][] is a digit representing gray color code in the 0-255 scale)
# y_train is a 1 dimensional array 60000 --> Each element is numeric representation of image in x_train
#
# x_test is a 3 dimensional array 10000 x 28 x 28 --> 10000 training images. Each image is of "28 x 28" pixel image showing hand-written digit sample.
# Each element in the x_test (i.e. x_test[][][] is a digit representing gray color code in the 0-255 scale)
# y_test is a 1 dimensional array 10000 --> Each element is numeric representation of image in x_test
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# -----------------------------------------------------------
# Step 2: Define preprocessing for images

# Reshape and normalize the numerical data
## It reshapes 3D array 60000 x 28 x 28 to 4D array 60000 x 28 x 28 x 1
## the last column represents the no of channels. Since our image is in gray scale, it has one channel. The last column stores 1 always.
## Why to 4D? Because the CNN architecture expects it in that format.
# x_train[]     --> Image Index (0 - 59999)
# x_train[][]   --> Image Pixel Row Index (0 - 27)
# x_train[][][] --> Image Pixel Column Index (0 - 27)
# x_train[][][][] --> Color Channel (1-3 for colored images, 1 for grayscaled images)
def reshape_and_scale(x):
    return x.reshape(-1, 28, 28, 1).astype("float32") / 255.0

# Wrap preprocessing inside sklearn FunctionTransformer
image_preprocessor = FunctionTransformer(reshape_and_scale)

# Normalize the categorical data (target --> supervised output that is a text data representing the digit.)
## Target here represents the hand-written digit class ranging from 0-9
## to_categorical() does one-hot encoding on it.
## One-hot encoding is a must for loss function "categorical_crossentropy"
y_train_enc = to_categorical(y_train)
y_test_enc = to_categorical(y_test)

# -----------------------------------------------------------
# Step 3: Define CNN model (to be wrapped inside SciKeras)
def create_cnn_model():
    model = models.Sequential([
        # Input Layer
        Input(shape=(28, 28, 1)),

        # Convolution Layer1 - Apply 32 filter/kernel of 3x3 size
        # Each image dimension is 28 x 28 x 1. Hence, the input layer should have 784 neurons to read each pixel.
        # input_shape=(28, 28, 1) defines the input layer neurons
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),  # Pooling using 2x2 filter

        # Convolution Layer2 - Apply 64 filter/kernel of 3x3 size
        ## input_shape() is not specified here. Keras determines the shape and passes it automatically.
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),  # Convert 2D array to 1D array input

        # Here comes the ANN layers
        layers.Dense(64, activation='relu'),    # ANN Hidden/Dense layer with 64 neurons
        layers.Dense(10, activation='softmax')  # 10 classes for MNIST (It is a multiclass of 10 predicting class 0 to 9 digit)
    ])

    # Compile the CNN model
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# -----------------------------------------------------------
# Step 4: Wrap CNN inside SciKeras KerasClassifier
## KerasClassifier is a wrapper class that allows you to use a Keras (TensorFlow) model
## just like a scikit-learn estimator (e.g., LogisticRegression, RandomForest).
keras_clf = KerasClassifier(
    model=create_cnn_model,
    epochs=8,
    batch_size=32,
    verbose=0,          # silence wrapper logs
    fit__verbose=1      # ✅ show epoch-by-epoch progress from Keras
)

# -----------------------------------------------------------
# Step 5: Build sklearn pipeline
pipeline = Pipeline([
    ("reshape_scale", image_preprocessor),  # reshape + normalize images
    ("clf", keras_clf)                      # CNN model
])

# -----------------------------------------------------------
# Step 6: Train the model
## Pass validation data explicitly using clf__validation_data
pipeline.fit(
    x_train, y_train_enc,
    clf__validation_data=(reshape_and_scale(x_test), y_test_enc)
)

# -----------------------------------------------------------
# Step 7: Evaluation

# ✅ Get trained Keras model from pipeline
model = pipeline.named_steps["clf"].model_

# Evaluate on training data
train_loss, train_acc = model.evaluate(reshape_and_scale(x_train), y_train_enc, verbose=0)
print("Training accuracy:", train_acc)

# Evaluate on test data
test_loss, test_acc = model.evaluate(reshape_and_scale(x_test), y_test_enc, verbose=0)
print("Test accuracy:", test_acc)

# -----------------------------------------------------------
# Step 8: Manual accuracy calculation

# 1. Predict probabilities for test data
pred = model.predict(reshape_and_scale(x_test))

# 2. Get predicted class labels by taking argmax
y_pred = tf.argmax(pred, axis=1).numpy()  # convert to NumPy array

# 3. If y_test is one-hot encoded, convert it back to class labels
if len(y_test_enc.shape) > 1 and y_test_enc.shape[1] > 1:
    y_true = tf.argmax(y_test_enc, axis=1).numpy()
else:
    y_true = y_test

# 4. Compute accuracy using sklearn
acc = accuracy_score(y_true, y_pred)
print(f"Test accuracy (manual): {acc:.4f}")


Epoch 1/4
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.8977 - loss: 0.3282 - val_accuracy: 0.9852 - val_loss: 0.0472
Epoch 2/4
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9856 - loss: 0.0472 - val_accuracy: 0.9879 - val_loss: 0.0346
Epoch 3/4
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.9911 - loss: 0.0281 - val_accuracy: 0.9893 - val_loss: 0.0337
Epoch 4/4
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9937 - loss: 0.0200 - val_accuracy: 0.9893 - val_loss: 0.0357
Training accuracy: 0.9954166412353516
Test accuracy: 0.989300012588501
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Test accuracy (manual): 0.9893


## A Simple CNN Code - Training on CIFAR10 Animal Dataset - Using Pipeline

In [5]:
# Step 1: Imports
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from matplotlib import pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from scikeras.wrappers import KerasClassifier
from sklearn.metrics import accuracy_score

# -----------------------------------------------------------
# Load and preprocess the data
# x_train is a 4D array: 50000 x 32 x 32 x 3 --> 50000 training images
# Each image is "32x32" pixels, with 3 color channels (RGB).
# y_train is a 1D array of 50000 --> Each element is a class label (0–9).
# x_test is 10000 x 32 x 32 x 3 --> 10000 test images.
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# -----------------------------------------------------------
# Step 2: Define preprocessing for images
# CIFAR-10 is already 4D (N, 32, 32, 3), so just scale values
def scale_images(x):
    return x.astype("float32") / 255.0

# Wrap preprocessing inside sklearn FunctionTransformer
image_preprocessor = FunctionTransformer(scale_images)

# One-hot encode labels
# Note: y_train/y_test are shape (N,1), so flatten before encoding
y_train_enc = to_categorical(y_train.flatten(), 10)
y_test_enc = to_categorical(y_test.flatten(), 10)

# -----------------------------------------------------------
# Step 3: Define CNN model (for CIFAR-10)
def create_cnn_model():
    model = models.Sequential([
        # Input Layer
        Input(shape=(32, 32, 3)),

        # Convolution + Pooling blocks
        layers.Conv2D(32, (3, 3), activation='relu', padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu', padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),

        # Fully connected layers
        layers.Dense(128, activation='sigmoid'),
        layers.Dense(10, activation='softmax')  # 10 CIFAR-10 classes
    ])

    # Compile CNN model
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# -----------------------------------------------------------
# Step 4: Wrap CNN inside SciKeras KerasClassifier
keras_clf = KerasClassifier(
    model=create_cnn_model,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=0,
    fit__verbose=1  # ✅ show epoch-by-epoch progress
)

# -----------------------------------------------------------
# Step 5: Build sklearn pipeline
pipeline = Pipeline([
    ("scale", image_preprocessor),  # normalize images
    ("clf", keras_clf)              # CNN model
])

# -----------------------------------------------------------
# Step 6: Train the model
pipeline.fit(
    x_train, y_train_enc,
    clf__validation_data=(scale_images(x_test), y_test_enc)
)

# -----------------------------------------------------------
# Step 7: Evaluation
model = pipeline.named_steps["clf"].model_

# Evaluate on training data
train_loss, train_acc = model.evaluate(scale_images(x_train), y_train_enc, verbose=0)
print("Training accuracy:", train_acc)

# Evaluate on test data
test_loss, test_acc = model.evaluate(scale_images(x_test), y_test_enc, verbose=0)
print("Test accuracy:", test_acc)

# -----------------------------------------------------------
# Step 8: Manual accuracy calculation
pred = model.predict(scale_images(x_test))  # probability distribution
y_pred = tf.argmax(pred, axis=1).numpy()

# Convert one-hot y_test_enc back to class labels
y_true = tf.argmax(y_test_enc, axis=1).numpy()

acc = accuracy_score(y_true, y_pred)
print(f"Test accuracy (manual): {acc:.4f}")


Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - accuracy: 0.3827 - loss: 1.6817 - val_accuracy: 0.6088 - val_loss: 1.0900
Epoch 2/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.6399 - loss: 1.0182 - val_accuracy: 0.6731 - val_loss: 0.9523
Epoch 3/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7133 - loss: 0.8197 - val_accuracy: 0.7050 - val_loss: 0.8328
Epoch 4/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7625 - loss: 0.6910 - val_accuracy: 0.7213 - val_loss: 0.8084
Epoch 5/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.7935 - loss: 0.5935 - val_accuracy: 0.7441 - val_loss: 0.7558
Epoch 6/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8254 - loss: 0.5127 - val_accuracy: 0.7483 - val_loss: 0.7486
Epoch 7/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8553 - loss: 0.4320 - val_accuracy: 0.7478 - val_loss: 0.7613
Epoch 8/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8720 - loss: 0.3722